In [2]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from PIL import Image, ImageDraw, ImageFont
import torchvision.transforms.v2 as transforms
import cv2
import numpy as np

In [3]:
# 1. Загрузка предобученной модели Faster R-CNN (обучена на датасете COCO - 90 классов)
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()  # Переводим модель в режим оценки/инференса

# Имена классов в датасете COCO (для отображения результатов)
COCO_INSTANCE_CATEGORY_NAMES = weights.meta["categories"]
COCO_INSTANCE_CATEGORY_NAMES[18] = 'Jerry'
model_2 = fasterrcnn_resnet50_fpn(weights=weights)


In [4]:
model_2.train()

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [5]:
print(COCO_INSTANCE_CATEGORY_NAMES)

['__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'Jerry', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


## Custom dataset

In [47]:
from torch.utils.data import Dataset

num_classes = 91

#model_2.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, targets, transforms=None):
        self.image_paths = image_paths
        self.targets = targets
        self.transforms = transforms

    def __getitem__(self, idx):
        # Загрузка изображения
        img = Image.open(r"C:\Users\user\Downloads\Telegram Desktop\\"+self.image_paths[idx]).convert("RGB")
        img = transforms.functional.to_image(img)
        img = transforms.functional.to_dtype(img, scale=True)
        target = self.targets[idx]
        
        if self.transforms is not None:
            img = self.transforms(img)
            
        return img, target

    def __len__(self):
        return len(self.image_paths)

image_paths = ["photo_2026-06-01_18-16-30.jpg", "photo_2026-06-01_18-16-19.jpg", "photo_2026-06-01_18-15-47.jpg", "barbos1.jpg", "barbos2.jpg", "barbos3.jpg", "barbos4.jpg", "barbos5.jpg", "barbos6.jpg", "barbos7.jpg", "barbos8.jpg"]

tgt = [
    { 'boxes' : torch.tensor([[485, 97, 606, 186]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[274, 184, 355, 339]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[395, 485, 592, 772]]),  
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[884, 356, 1185, 644]]),  
      'labels' : torch.tensor([18])
    }
      ,
    { 'boxes' : torch.tensor([[935, 384, 1085, 682]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[748, 410, 1156, 636]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[728, 411, 1137, 683]]),  
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[686, 424, 1086, 811]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[901, 433, 1093, 684]]),  
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[940, 443, 1113, 674]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[828, 306, 1136, 560]]),  
      'labels' : torch.tensor([18])
    }
]

dog_fine = CustomDataset(image_paths, tgt)
print(dog_fine[0])

(Image([[[0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        ...,
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.]],

       [[0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        ...,
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.]],

       [[0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        ...,
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.]]], ), {'boxes': tensor([[485,  97, 606, 186]]), 'labels': tensor([18])})


## Training

In [48]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model_2.to(device)

# Параметры оптимизатора
params = [p for p in model_2.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Функция для правильного объединения словарей в батч
def collate_fn(batch):
    return tuple(zip(*batch))

model_2.train()

data_loader = torch.utils.data.DataLoader(
    dog_fine, batch_size=2, shuffle=True, collate_fn=collate_fn
)

for epoch in range(50): # количество эпох
    for images, targets in data_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model_2(images, targets)
        #print(type(loss_dict))
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
    print(f"Epoch {epoch} loss: {losses.item()}")


Epoch 0 loss: 0.00019336638797540218
Epoch 1 loss: 0.0001097795320674777
Epoch 2 loss: 7.359417213592678e-05
Epoch 3 loss: 0.0003223399689886719
Epoch 4 loss: 0.0006150731933303177
Epoch 5 loss: 0.0003887080238200724
Epoch 6 loss: 6.759921234333888e-05
Epoch 7 loss: 0.00038504914846271276
Epoch 8 loss: 0.00023899204097688198
Epoch 9 loss: 6.41873775748536e-05
Epoch 10 loss: 0.00037752409116365016
Epoch 11 loss: 0.00011012046888936311
Epoch 12 loss: 0.00017629536159802228
Epoch 13 loss: 0.00015468346828129143
Epoch 14 loss: 0.00022152200108394027
Epoch 15 loss: 0.00012026912736473605
Epoch 16 loss: 0.0001021097632474266
Epoch 17 loss: 0.0003012105589732528
Epoch 18 loss: 0.00012565618089865893
Epoch 19 loss: 0.0003499477752484381
Epoch 20 loss: 0.00014559432747773826
Epoch 21 loss: 0.000161794523592107
Epoch 22 loss: 0.000147737460792996
Epoch 23 loss: 0.00015887957124505192
Epoch 24 loss: 7.856345473555848e-05
Epoch 25 loss: 0.0002748646365944296
Epoch 26 loss: 6.436525291064754e-05
Ep

## Testing

In [49]:
image_path = r"C:\Users\user\Downloads\Telegram Desktop"
image_path += r"\photo_2026-06-01_18-15-15.jpg"  # Укажите путь к вашей картинке
img = Image.open(image_path).convert("RGB")

# Преобразуем картинку в тензор PyTorch и добавляем размерность батча [1, C, H, W]
img_tensor = F.to_tensor(img).unsqueeze(0)

model_2.eval()
# 3. Поиск объектов (Получение предсказаний)
with torch.no_grad():
    predictions = model_2(img_tensor)

# Разбираем результаты (берем первый элемент батча)
boxes = predictions[0]['boxes']      # Координаты рамок [xmin, ymin, xmax, ymax]
labels = predictions[0]['labels']    # Индексы классов объектов
scores = predictions[0]['scores']    # Уверенность модели в предсказании (0.0 - 1.0)

draw = ImageDraw.Draw(img)
threshold = .8# Оставляем только объекты, в которых сеть уверена более чем на 80%
for box, label, score in zip(boxes, labels, scores):
    class_name = COCO_INSTANCE_CATEGORY_NAMES[label]
    #print(class_name)
    if score > threshold and class_name=='Jerry':
        xmin, ymin, xmax, ymax = box.tolist()
        #print(class_name)
        draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=3)
        draw.text((xmin, ymin - 30), f"{class_name}: {score*100:.0f} %", fill="red", size = 40, font=ImageFont.truetype("arial.ttf", size=28))
img.show()

## Video

In [56]:
model_2.rpn.nms_thresh = 0.1
model_2.roi_heads.nms_thresh = 0.01
model_2.eval()
def video_to_frames(video_path, frame_rate):

    cap = cv2.VideoCapture(video_path)
    
    frame_count = 0
    saved_count = 0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    while cap.isOpened():
        
        ret, frame = cap.read()
        if not ret:
            break  
        if frame_count % frame_rate == 0:
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_image)
            img_tensor = F.to_tensor(pil_image).unsqueeze(0)

            with torch.no_grad():
                predictions = model_2(img_tensor)

            boxes = predictions[0]['boxes']     
            labels = predictions[0]['labels']    
            scores = predictions[0]['scores']    

            draw = ImageDraw.Draw(pil_image)
            threshold = .2
            for box, label, score in zip(boxes, labels, scores):
                class_name = COCO_INSTANCE_CATEGORY_NAMES[label]
                if score > threshold and class_name=='Jerry':
                    xmin, ymin, xmax, ymax = box.tolist()
                    draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=3)
                    draw.text((xmin, ymin - 30), f"{'Jerry'}: {score*100:.0f} %", fill="red", size = 40, font=ImageFont.truetype("arial.ttf", size=28))
            
            #pil_image.show()
            rgb_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_BGR2RGB)
            #cv2_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
            height, width, layers = rgb_image.shape
            if frame_count == 0:
                video = cv2.VideoWriter(r"C:\Users\user\Downloads\Telegram Desktop\jerry_cv.mp4", fourcc, 30, (width, height), isColor=True)
            video.write(rgb_image)
            #cv2.imshow('hui', rgb_image)
            #cv2.waitKey(0)
            #cv2.destroyAllWindows()
            saved_count += 1
            print('frame', saved_count*frame_rate)
            
        frame_count += 1

    cap.release()
    video.release()
    #cv2.waitKey(0)
    #cv2.destroyAllWindows()
    print(f"Готово! Извлечено кадров: {saved_count} из {frame_count}")

video_path = r"C:\Users\user\Downloads\Telegram Desktop"
video_path += r"\jerry_2_2.mp4"
video_to_frames(video_path, frame_rate=1)


frame 1
frame 2
frame 3
frame 4
frame 5
frame 6
frame 7
frame 8
frame 9
frame 10
frame 11
frame 12
frame 13
frame 14
frame 15
frame 16
frame 17
frame 18
frame 19
frame 20
frame 21
frame 22
frame 23
frame 24
frame 25
frame 26
frame 27
frame 28
frame 29
frame 30
frame 31
frame 32
frame 33
frame 34
frame 35
frame 36
frame 37
frame 38
frame 39
frame 40
frame 41
frame 42
frame 43
frame 44
frame 45
frame 46
frame 47
frame 48
frame 49
frame 50
frame 51
frame 52
frame 53
frame 54
frame 55
frame 56
frame 57
frame 58
frame 59
frame 60
frame 61
frame 62
frame 63
frame 64
frame 65
frame 66
frame 67
frame 68
frame 69
frame 70
frame 71
frame 72
frame 73
frame 74
frame 75
frame 76
frame 77
frame 78
frame 79
frame 80
frame 81
frame 82
frame 83
frame 84
frame 85
frame 86
frame 87
frame 88
frame 89
frame 90
frame 91
frame 92
frame 93
frame 94
frame 95
frame 96
frame 97
frame 98
frame 99
frame 100
frame 101
frame 102
frame 103
frame 104
frame 105
frame 106
frame 107
frame 108
frame 109
frame 110
frame 11